In [1]:
# takes cleaned CSV and turns it into a real SQL database table you can query
import sqlite3
import pandas as pd

# Load the cleaned CSV
df = pd.read_csv("superstore_cleaned.csv", parse_dates=["Order Date", "Ship Date"])

# Create (or connect to) a SQLite database file
conn = sqlite3.connect("superstore.db")

# Write the DataFrame into a SQL table called "orders"
df.to_sql("orders", conn, if_exists="replace", index=False)

print("Loaded successfully")

Loaded successfully


In [2]:
# Testing
query = "SELECT * FROM orders LIMIT 5;"
pd.read_sql(query, conn)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,2016-11-08 00:00:00,2016-11-11 00:00:00,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,2016-11-08 00:00:00,2016-11-11 00:00:00,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,2016-06-12 00:00:00,2016-06-16 00:00:00,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,2015-10-11 00:00:00,2015-10-18 00:00:00,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,2015-10-11 00:00:00,2015-10-18 00:00:00,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [3]:
# Triple quotes (""") let me write the SQL query as one string across multiple lines.
# The query is stored as text here; pd.read_sql(query, conn) runs it below.

query = """
SELECT
    Category,
    SUM(Sales) AS Total_Sales,
    SUM(Profit) AS Total_Profit,
    ROUND(SUM(Profit) * 100.0 / SUM(Sales), 2) AS Profit_Margin_Pct,
    ROUND(AVG(Discount) * 100, 2) AS Avg_Discount_Pct
FROM orders
GROUP BY Category
ORDER BY Total_Profit DESC;
"""
pd.read_sql(query, conn)

,Category,Total_Sales,Total_Profit,Profit_Margin_Pct,Avg_Discount_Pct
0,Technology,836154.0330,145454.9481,17.40,13.23
1,Office Supplies,719047.0320,122490.8008,17.04,15.73
2,Furniture,741999.7953,18451.2728,2.49,17.39


In [4]:
query = """
SELECT
    "Sub-Category",
    SUM(Sales) AS Total_Sales,
    SUM(Profit) AS Total_Profit,
    ROUND(SUM(Profit) * 100.0 / SUM(Sales), 2) AS Profit_Margin_Pct,
    ROUND(AVG(Discount) * 100, 2) AS Avg_Discount_Pct
FROM orders
WHERE Category = 'Furniture'
GROUP BY "Sub-Category"
ORDER BY Profit_Margin_Pct ASC;
"""
pd.read_sql(query, conn)

,Sub-Category,Total_Sales,Total_Profit,Profit_Margin_Pct,Avg_Discount_Pct
0,Tables,206965.5320,-17725.4811,-8.56,26.13
1,Bookcases,114879.9963,-3472.5560,-3.02,21.11
2,Chairs,328449.1030,26590.1663,8.10,17.02
3,Furnishings,91705.1640,13059.1436,14.24,13.83


In [5]:
query = """
SELECT
    Region,
    SUM(Sales) AS Total_Sales,
    SUM(Profit) AS Total_Profit,
    ROUND(SUM(Profit) * 100.0 / SUM(Sales), 2) AS Profit_Margin_Pct,
    ROUND(AVG(Discount) * 100, 2) AS Avg_Discount_Pct
FROM orders
GROUP BY Region
ORDER BY Profit_Margin_Pct ASC;
"""
pd.read_sql(query, conn)

,Region,Total_Sales,Total_Profit,Profit_Margin_Pct,Avg_Discount_Pct
0,Central,501239.8908,39706.3625,7.92,24.04
1,South,391721.9050,46749.4303,11.93,14.73
2,East,678781.2400,91522.7800,13.48,14.54
3,West,725457.8245,108418.4489,14.94,10.93


In [6]:
query = """
SELECT
    Segment,
    SUM(Sales) AS Total_Sales,
    SUM(Profit) AS Total_Profit,
    ROUND(SUM(Profit) * 100.0 / SUM(Sales), 2) AS Profit_Margin_Pct,
    ROUND(AVG(Discount) * 100, 2) AS Avg_Discount_Pct
FROM orders
GROUP BY Segment
ORDER BY Profit_Margin_Pct ASC;
"""
pd.read_sql(query, conn)

,Segment,Total_Sales,Total_Profit,Profit_Margin_Pct,Avg_Discount_Pct
0,Consumer,1.161401e+06,134119.2092,11.55,15.81
1,Corporate,7.061464e+05,91979.1340,13.03,15.82
2,Home Office,4.296531e+05,60298.6785,14.03,14.71


In [7]:
query = """
SELECT
    Segment,
    Category,
    SUM(Sales) AS Total_Sales,
    ROUND(SUM(Sales) * 100.0 / SUM(SUM(Sales)) OVER (PARTITION BY Segment), 2) AS Pct_Of_Segment_Sales,
    ROUND(SUM(Profit) * 100.0 / SUM(Sales), 2) AS Profit_Margin_Pct
FROM orders
GROUP BY Segment, Category
ORDER BY Segment, Pct_Of_Segment_Sales DESC;
"""
pd.read_sql(query, conn)

,Segment,Category,Total_Sales,Pct_Of_Segment_Sales,Profit_Margin_Pct
0,Consumer,Technology,406399.8970,34.99,17.42
1,Consumer,Furniture,391049.3120,33.67,1.79
2,Consumer,Office Supplies,363952.1360,31.34,15.48
3,Corporate,Technology,246450.1190,34.90,17.92
4,Corporate,Office Supplies,230676.4620,32.67,17.44
5,Corporate,Furniture,229019.7858,32.43,3.31
6,Home Office,Technology,183304.0170,42.66,16.63
7,Home Office,Office Supplies,124418.4340,28.96,20.84
8,Home Office,Furniture,121930.6975,28.38,3.18


In [8]:
query = """
SELECT
    Segment,
    Category,
    SUM(Sales) AS Total_Sales,
    ROUND(SUM(Profit) * 100.0 / SUM(Sales), 2) AS Profit_Margin_Pct,
    ROUND(AVG(Discount) * 100, 2) AS Avg_Discount_Pct
FROM orders
WHERE Category = 'Furniture'
GROUP BY Segment
ORDER BY Avg_Discount_Pct DESC;
"""
pd.read_sql(query, conn)

,Segment,Category,Total_Sales,Profit_Margin_Pct,Avg_Discount_Pct
0,Consumer,Furniture,391049.3120,1.79,17.67
1,Corporate,Furniture,229019.7858,3.31,17.41
2,Home Office,Furniture,121930.6975,3.18,16.50
